In [ ]:
!pip install -q medmnist

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset
from medmnist import INFO, DermaMNIST
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import time, random
import matplotlib.pyplot as plt

# Enable PyTorch memory optimization
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# --- Configuration ---
config = {
    'num_clients': 5,
    'batch_size': 64,
    'epochs': 5,
    'unlearn_epochs': 3,
    'post_train_epochs': 2,
    'lr': 1e-4,
    'alpha': 1.0
}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# --- Load DermaMNIST Dataset ---
info = INFO['dermamnist']
data_class = DermaMNIST
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
])

train_dataset = data_class(split='train', transform=transform, download=True)
test_dataset = data_class(split='test', transform=transform, download=True)

In [ ]:
# Split train into clients
indices = np.arange(len(train_dataset))
np.random.shuffle(indices)
proportions = np.random.dirichlet(np.repeat(config['alpha'], config['num_clients']))
client_datasets = []
start = 0
for p in proportions:
    num_samples = int(p * len(indices))
    client_indices = indices[start:start+num_samples]
    client_datasets.append(Subset(train_dataset, client_indices))
    start += num_samples

In [ ]:
# --- Model ---
def get_densenet121(num_classes=7):
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model

In [ ]:
# --- Training & Evaluation ---
def train_model(model, loader, optimizer, epochs):
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze().long().to(DEVICE)
            loss = F.cross_entropy(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            del x, y
            torch.cuda.empty_cache()

def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze().cpu()
            preds = model(x).argmax(1).cpu()
            y_true.extend(y)
            y_pred.extend(preds)
            del x
    return accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average='macro')

def error_metrics(model, loaders):
    error_f, error_r = 1.0, 0.0
    for i, loader in enumerate(loaders):
        acc, _ = evaluate(model, loader)
        if i == 0:
            error_f = 1 - acc
        else:
            error_r += (1 - acc)
    error_r /= (len(loaders) - 1)
    return error_f, error_r

In [ ]:
# --- MCU ---
def model_contrastive_unlearning(model, trained_model, downgraded_model, loader, epochs, tau=0.5):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    cos = nn.CosineSimilarity(dim=1)
    for _ in range(epochs):
        for x, _ in loader:
            x = x.to(DEVICE)
            z = model.features(x).mean(dim=(2, 3))
            z_tr = trained_model.features(x).mean(dim=(2, 3)).detach()
            z_down = downgraded_model.features(x).mean(dim=(2, 3)).detach()
            sim_pos = cos(z, z_down)
            sim_neg = cos(z, z_tr)
            logits = torch.stack([sim_pos, sim_neg], dim=1)
            labels = torch.zeros(len(z), dtype=torch.long, device=DEVICE)
            loss = F.cross_entropy(logits / tau, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            del x, z, z_tr, z_down
            torch.cuda.empty_cache()
    return model

In [ ]:
# --- FGMP ---
def fft_fusion_full(w_pre, w_adj, r=0.2):
    shape = w_pre.shape
    if w_pre.ndim < 2:
        return w_adj
    w_pre = w_pre.detach().cpu().numpy()
    w_adj = w_adj.detach().cpu().numpy()
    W1 = w_pre.reshape(shape[0], -1)
    W2 = w_adj.reshape(shape[0], -1)
    F1 = np.fft.fft2(W1)
    F2 = np.fft.fft2(W2)
    mask = np.zeros_like(F1)
    center = (F1.shape[0] // 2, F1.shape[1] // 2)
    r1 = int(r * F1.shape[0] / 2)
    r2 = int(r * F1.shape[1] / 2)
    mask[center[0]-r1:center[0]+r1, center[1]-r2:center[1]+r2] = 1
    fused = mask * F1 + (1 - mask) * F2
    fused = np.fft.ifft2(fused).real
    fused = fused.reshape(shape)
    return torch.tensor(fused, dtype=torch.float32, device=DEVICE)

In [ ]:
# --- Federated Averaging ---
def federated_avg(models, lens):
    global_model = get_densenet121().to(DEVICE)
    global_dict = global_model.state_dict()
    for k in global_dict.keys():
        global_dict[k] = sum(m.state_dict()[k] * l for m, l in zip(models, lens)) / sum(lens)
    global_model.load_state_dict(global_dict)
    return global_model

In [ ]:
# --- Run FCU ---
print("\n=== FCU on DermaMNIST ===")
client_loaders = [DataLoader(ds, batch_size=config['batch_size'], shuffle=True) for ds in client_datasets]
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'])

In [ ]:
# Initial federated training
models_trained = []
for i, loader in enumerate(client_loaders):
    print(f"Training client {i}...")
    model = get_densenet121().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    train_model(model, loader, optimizer, config['epochs'])
    models_trained.append(model)

lens = [len(ds) for ds in client_datasets]
global_model = federated_avg(models_trained, lens)
acc, f1 = evaluate(global_model, test_loader)
print(f"Initial Global Model - Acc: {acc:.4f}, F1: {f1:.4f}")

In [ ]:
# Error before unlearning
error_f_before, error_r_before = error_metrics(global_model, client_loaders)

In [ ]:
# Target client 0 unlearning
target_model = get_densenet121().to(DEVICE)
target_model.load_state_dict(global_model.state_dict())
downgraded_model = get_densenet121().to(DEVICE)

print("\nUnlearning client 0 with MCU...")
unlearned_model = model_contrastive_unlearning(target_model, global_model, downgraded_model, client_loaders[0], config['unlearn_epochs'])

print("Applying FGMP...")
with torch.no_grad():
    un_state = unlearned_model.state_dict()
    gl_state = global_model.state_dict()
    for name in un_state:
        if name in gl_state and un_state[name].ndim >= 2:
            fused = fft_fusion_full(gl_state[name], un_state[name])
            un_state[name].copy_(fused)
global_model.load_state_dict(un_state)

In [ ]:
# Error after MCU+FGMP
error_f_after_mcu, error_r_after_mcu = error_metrics(global_model, client_loaders)

In [ ]:
# Post-training with other clients
post_models, post_lens = [], []
for i in range(1, config['num_clients']):
    print(f"Post-training client {i}...")
    model = get_densenet121().to(DEVICE)
    model.load_state_dict(global_model.state_dict())
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    train_model(model, client_loaders[i], optimizer, config['post_train_epochs'])
    post_models.append(model)
    post_lens.append(len(client_datasets[i]))

global_model = federated_avg(post_models, post_lens)
acc, f1 = evaluate(global_model, test_loader)
print(f"\nFinal Unlearned Model - Acc: {acc:.4f}, F1: {f1:.4f}")

In [ ]:
# Final error metrics
error_f_final, error_r_final = error_metrics(global_model, client_loaders)

In [ ]:
# Plot Error_f and Error_r over stages
stages = ['Before Unlearning', 'After MCU', 'Post-Training']
error_f_values = [error_f_before, error_f_after_mcu, error_f_final]
error_r_values = [error_r_before, error_r_after_mcu, error_r_final]

plt.figure(figsize=(10, 5))
plt.plot(stages, error_f_values, marker='o', label='Error_f (Forgotten Client)')
plt.plot(stages, error_r_values, marker='s', label='Error_r (Remaining Clients)')
plt.title('Error_f and Error_r Over Unlearning Stages')
plt.xlabel('Stage')
plt.ylabel('Error')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('error_metrics_plot.png')
plt.show()

print("\n📈 Error Metrics Summary")
print(f"Error_f Before: {error_f_before:.4f}, After MCU: {error_f_after_mcu:.4f}, Final: {error_f_final:.4f}")
print(f"Error_r Before: {error_r_before:.4f}, After MCU: {error_r_after_mcu:.4f}, Final: {error_r_final:.4f}")